# Mô hình: Attention CNN 1D (CBAM)
**Dataset**: Diabetes
**Bài tập**: Thực nghiệm CNN 1D trên dữ liệu Tabular


**Học viện Công nghệ Bưu chính Viễn thông (PTIT) — Khoa CNTT 1**
- **Sinh viên:** Nguyễn Nam Hải (B23DCCN277 - D23CTPM01 - CT01)
- **GVHD:** PGS.TS. Trần Đình Quế
- **GitHub Repository:** [https://github.com/HandQ2212/intel-sys-assignment-05](https://github.com/HandQ2212/intel-sys-assignment-05)
- **Kaggle Dataset:** [Diabetes Health Indicators Dataset (Binary Classification)](https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset)


## 2. Cơ sở lý thuyết: Attention CNN 1D (CBAM)
Cơ chế Attention (như CBAM - Convolutional Block Attention Module) giúp mô hình tập trung vào những đặc trưng quan trọng nhất.
Với 1D, CBAM bao gồm 2 phần:
1. **Channel Attention**: Nhấn mạnh các channel quan trọng (các bộ lọc tạo ra phản ứng mạnh).
   $$ M_c(F) = \sigma(MLP(AvgPool1D(F)) + MLP(MaxPool1D(F))) $$
2. **Temporal Attention** (thay cho Spatial trong 2D): Nhấn mạnh vị trí quan trọng trong chuỗi đặc trưng 1D.
   $$ M_s(F) = \sigma(Conv1D([AvgPool1D(F); MaxPool1D(F)])) $$
   
Ứng dụng trên dữ liệu tabular là để quan sát xem liệu việc kết hợp Attention có bù đắp được sự thiếu vắng về cấu trúc không gian tự nhiên của dữ liệu không.


In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
EPOCHS = 20
BATCH_SIZE = 256
LR = 1e-3
IN_CHANNELS = 1
NUM_CLASSES = 1
DATASET_NAME = 'Diabetes'
MODEL_NAME = 'Attention CNN 1D (CBAM)'
MODEL_KEY = 'attention'


## 3. Data Loading & Preprocessing


In [ ]:
df = pd.read_csv('../intel-sys-assignment-04/dataset/diabets.csv')
# Lấy mẫu 50000 dòng để tính toán nhanh hơn
df = df.sample(n=50000, random_state=42)

X = df.drop(columns=['Diabetes_binary']).values
y = df['Diabetes_binary'].values

# Standard scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Chuyển thành tensor và thêm chiều channel (batch, channel, features) -> (batch, 1, 21)
X_train_tensor = torch.FloatTensor(X_train).unsqueeze(1)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test).unsqueeze(1)
y_test_tensor = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


## 4. Data Exploration


In [ ]:
print("Head of dataset:")
display(df.head())
print("\nDataset description:")
display(df.describe())


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Diabetes_binary')
plt.title("Class Distribution")
plt.show()


In [ ]:
plt.figure(figsize=(15, 12))
sns.heatmap(df.corr(), annot=False, cmap='coolwarm')
plt.title("Feature Correlation")
plt.show()


## 5. Model Architecture


In [ ]:
class ChannelAttention1D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        hidden = max(channels // reduction, 2)
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden, bias=False), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        b, c, _ = x.shape
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        return x * self.sigmoid(avg_out + max_out).view(b, c, 1)

class TemporalAttention1D(nn.Module):
    def __init__(self, kernel_size=3):
        super().__init__()
        self.conv = nn.Conv1d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM1D(nn.Module):
    def __init__(self, channels, reduction=4, temporal_kernel=3):
        super().__init__()
        self.ca = ChannelAttention1D(channels, reduction)
        self.ta = TemporalAttention1D(temporal_kernel)
    def forward(self, x):
        return self.ta(self.ca(x))

class AttentionCNN1D(nn.Module):
    def __init__(self, in_channels=1, num_classes=1):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv1d(in_channels, 32, 3, padding=1, bias=False), nn.BatchNorm1d(32), nn.ReLU(inplace=True))
        self.cbam1 = CBAM1D(32, reduction=4)
        self.pool1 = nn.MaxPool1d(2)
        self.block2 = nn.Sequential(nn.Conv1d(32, 64, 3, padding=1, bias=False), nn.BatchNorm1d(64), nn.ReLU(inplace=True))
        self.cbam2 = CBAM1D(64, reduction=4)
        self.pool2 = nn.MaxPool1d(2)
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(64, num_classes))
    def forward(self, x):
        x = self.pool1(self.cbam1(self.block1(x)))
        x = self.pool2(self.cbam2(self.block2(x)))
        return self.classifier(x)

model = AttentionCNN1D(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES).to(device)
print(model)


## 6. Training Functions


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs).squeeze(-1)  # (batch, 1) -> (batch,)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * inputs.size(0)
        predicted = (torch.sigmoid(outputs) >= 0.5).float()
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs).squeeze(-1)
        loss = criterion(outputs, labels)
        
        total_loss += loss.item() * inputs.size(0)
        probs = torch.sigmoid(outputs)
        predicted = (probs >= 0.5).float()
        
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        
    return total_loss / total, correct / total, all_preds, all_labels, all_probs


## 7. Training Loop


In [ ]:
train_losses, test_losses = [], []
train_accs, test_accs = [], []

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, _, _, _ = evaluate(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f}, Acc: {test_acc:.4f}")


## 8. Visualization


In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train')
plt.plot(test_losses, label='Test')
plt.title('Loss over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train')
plt.plot(test_accs, label='Test')
plt.title('Accuracy over Epochs')
plt.legend()
plt.show()


## 9. Final Evaluation


In [ ]:
test_loss, test_acc, all_preds, all_labels, all_probs = evaluate(model, test_loader, criterion, device)

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['No Diabetes', 'Diabetes']))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Diabetes', 'Diabetes'], yticklabels=['No Diabetes', 'Diabetes'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.4f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()


## 10. Save Results


In [ ]:
os.makedirs('results', exist_ok=True)
results = {
    'model_name': MODEL_NAME,
    'dataset': DATASET_NAME,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LR,
    'final_train_acc': train_accs[-1],
    'final_test_acc': test_accs[-1],
    'final_train_loss': train_losses[-1],
    'final_test_loss': test_losses[-1],
    'roc_auc': roc_auc
}

res_path = f"results/{DATASET_NAME.lower()}_{MODEL_KEY}.json"
with open(res_path, 'w') as f:
    json.dump(results, f, indent=4)
print(f"Saved results to {res_path}")


## 11. Conclusion
Mô hình Attention CNN 1D (CBAM) đã được huấn luyện và đánh giá trên tập dữ liệu Diabetes (dữ liệu bảng coi như chuỗi 1D).
